# UCS420 Cognitive Computing — Assignment 10
**Diabetes Care Information Assistant**  
**Student:** Diksha Mahajan  
**Roll Number:** 1024160059 (Odd)  
**Platform:** Google Colab

Educational information only; not a diagnosis or substitute for professional medical care.

In [ ]:
import re, html, nltk, numpy as np, pandas as pd, matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity
for pkg in ['stopwords','wordnet','omw-1.4']:
    nltk.download(pkg, quiet=True)
print('Environment ready')

In [ ]:
# Requirement 1: 60 labelled examples across six intents
DATA = {
'symptoms': [
'What are common symptoms of diabetes?', 'Why am I thirsty all the time?', 'Does frequent urination indicate diabetes?', 'What are signs of high blood sugar?', 'Can diabetes cause blurry vision?', 'Why am I tired with high glucose?', 'Does diabetes cause unexplained weight loss?', 'What are early warning signs of diabetes?', 'Can cuts heal slowly in diabetes?', 'What symptoms should I watch for with diabetes?'],
'risk_factors': [
'Who is at risk of type 2 diabetes?', 'Does family history increase diabetes risk?', 'Can inactivity contribute to diabetes?', 'How does age affect diabetes risk?', 'Does gestational diabetes increase future risk?', 'What are risk factors for diabetes?', 'Can high blood pressure be linked to diabetes risk?', 'Does prediabetes increase the chance of diabetes?', 'Is type 2 diabetes more likely with a family history?', 'How does physical activity affect diabetes risk?'],
'prevention': [
'How can I reduce my risk of type 2 diabetes?', 'What healthy habits help prevent diabetes?', 'Does regular walking lower diabetes risk?', 'How can balanced meals help prevent diabetes?', 'Why is exercise useful for diabetes prevention?', 'What lifestyle changes reduce type 2 diabetes risk?', 'Can reducing sugary drinks help prevent diabetes?', 'How does sleep support healthy blood sugar?', 'What foods are part of diabetes prevention?', 'How do I build a diabetes prevention routine?'],
'nutrition': [
'What can someone with diabetes eat?', 'How do carbohydrates affect blood sugar?', 'Are vegetables good for diabetes meals?', 'Can I eat fruit if I have diabetes?', 'How should I plan a balanced diabetes plate?', 'What is the role of fibre in diabetes?', 'How can I choose drinks with diabetes?', 'What is carbohydrate counting?', 'Should I skip meals to control glucose?', 'How do portion sizes affect blood sugar?'],
'monitoring': [
'How is diabetes diagnosed?', 'What does an A1C test measure?', 'What is fasting blood glucose?', 'How do glucose meters work?', 'What does a continuous glucose monitor do?', 'Why check blood sugar levels?', 'How often should glucose be checked?', 'What is an oral glucose tolerance test?', 'How should I record glucose readings?', 'What do lab blood sugar results mean?'],
'medications': [
'What medicines are used for diabetes?', 'What is insulin used for?', 'What does metformin do?', 'Can I stop taking my diabetes medication?', 'How do diabetes medicines lower glucose?', 'What if I miss a diabetes medicine dose?', 'Should I change my insulin dose myself?', 'What side effects should I discuss with a pharmacist?', 'Do all people with diabetes need insulin?', 'How can I manage my prescribed diabetes treatment?']
}
rows = [(q, intent) for intent, questions in DATA.items() for q in questions]
df = pd.DataFrame(rows, columns=['query','intent'])
print('Examples:',len(df),'Intents:',df.intent.nunique()); display(df.groupby('intent').size().rename('count').to_frame()); display(df.head())

In [ ]:
# Requirement 2: hierarchical taxonomy (4 major categories, >=2 subcategories each)
TAXONOMY = {
 'Understanding Diabetes': ['Symptoms and warning signs','Types and risk factors'],
 'Daily Care': ['Nutrition and meal planning','Movement and prevention'],
 'Testing and Monitoring': ['Diagnosis and lab tests','Home glucose monitoring'],
 'Treatment and Safety': ['Medications and insulin','Urgent warning signs and emergency response']
}
for category, subcategories in TAXONOMY.items():
    print(category)
    for sub in subcategories: print('  └─',sub)

In [ ]:
# Requirement 3: domain concepts and >=10 relationships
CONCEPTS = ['Diabetes','Blood Glucose','Insulin','Pancreas','Type 1 Diabetes','Type 2 Diabetes',
            'A1C','Glucose Meter','Carbohydrates','Physical Activity','Metformin','Hypoglycemia','Hyperglycemia']
RELATIONSHIPS = [
 ('Pancreas','produces','Insulin'), ('Insulin','helps regulate','Blood Glucose'),
 ('Diabetes','affects','Blood Glucose'), ('Type 1 Diabetes','is a type of','Diabetes'),
 ('Type 2 Diabetes','is a type of','Diabetes'), ('A1C','reflects average','Blood Glucose'),
 ('Glucose Meter','measures','Blood Glucose'), ('Carbohydrates','influence','Blood Glucose'),
 ('Physical Activity','can influence','Blood Glucose'), ('Metformin','can help manage','Blood Glucose'),
 ('Hypoglycemia','means low','Blood Glucose'), ('Hyperglycemia','means high','Blood Glucose')]
print('Concepts:',len(CONCEPTS),'Relationships:',len(RELATIONSHIPS))
for a,r,b in RELATIONSHIPS: print(f'{a} --{r}--> {b}')

In [ ]:
# Requirement 4: lowercase, tokenization, stopwords, lemmatization
tokenizer = RegexpTokenizer(r'[a-z]+')
STOP = set(stopwords.words('english')) - {'no','not'}
lemmatizer = WordNetLemmatizer()
def preprocess(text):
    tokens = tokenizer.tokenize(text.lower())
    return [lemmatizer.lemmatize(lemmatizer.lemmatize(t,'n'),'v') for t in tokens if t not in STOP]
def clean(text): return ' '.join(preprocess(text))
df['clean'] = df['query'].apply(clean)
example = 'What are the early symptoms of Diabetes?'
print('Original:',example,'\nLowercase:',example.lower(),'\nTokens:',tokenizer.tokenize(example.lower()),'\nProcessed:',preprocess(example))

In [ ]:
# Requirement 5: TF-IDF + supervised intent classifiers
X_train, X_test, y_train, y_test = train_test_split(df.clean, df.intent, test_size=0.30, random_state=42, stratify=df.intent)
vectorizer = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True)
Xtr = vectorizer.fit_transform(X_train); Xte = vectorizer.transform(X_test)
models = {'Naive Bayes':MultinomialNB(alpha=0.5), 'Logistic Regression':LogisticRegression(max_iter=2000,C=10)}
for name, model in models.items():
    model.fit(Xtr,y_train)
    print(name, 'held-out accuracy:',round(accuracy_score(y_test,model.predict(Xte)),3))
clf = models['Logistic Regression']
print('Training samples:',len(X_train),'Test samples:',len(X_test))

In [ ]:
# Requirement 6: held-out evaluation (do not claim perfect performance)
pred = clf.predict(Xte)
precision, recall, f1, _ = precision_recall_fscore_support(y_test,pred,average='weighted',zero_division=0)
print('Accuracy:',round(accuracy_score(y_test,pred),3))
print('Precision:',round(precision,3),'Recall:',round(recall,3),'F1:',round(f1,3))
print('\n',classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,xticks_rotation=45,cmap='Blues')
plt.title('Diabetes Assistant — Confusion Matrix'); plt.tight_layout(); plt.show()

In [ ]:
# Requirement 7: diabetes-specific knowledge base, one or more entries per intent
KB = {
'symptoms': [('symptoms thirst urination tired blurry vision', 'Possible symptoms include frequent urination, increased thirst, fatigue, blurry vision and unexplained weight changes. Symptoms alone cannot confirm diabetes; ask a clinician about testing.'),('slow healing wounds diabetes symptoms','Slow-healing wounds may occur with diabetes, but other conditions can also cause them. Seek clinical assessment.')],
'risk_factors': [('family history age inactivity prediabetes risk', 'Type 2 diabetes risk can be associated with family history, prediabetes, past gestational diabetes and lower physical activity. A clinician can help assess individual risk.')],
'prevention': [('prevent diabetes physical activity sleep habits', 'Regular enjoyable physical activity, balanced meals, adequate sleep and routine checkups can help lower the risk of type 2 diabetes. Type 1 diabetes is not currently preventable.')],
'nutrition': [('food carbohydrates fibre fruit vegetables balanced plate drinks', 'A balanced eating pattern can include vegetables, fruit, whole grains, protein and fibre. Carbohydrates affect glucose, but people with diabetes do not necessarily need to avoid them. Individual meal planning can be discussed with a registered dietitian.'),('skip meals glucose diabetes', 'Skipping meals may be unsafe for some people taking glucose-lowering medicines. Ask your care team for a meal schedule suited to your treatment.')],
'monitoring': [('a1c fasting glucose tolerance diagnosis test', 'Diabetes can be assessed using laboratory tests such as A1C, fasting plasma glucose or an oral glucose tolerance test. A clinician interprets results and confirms a diagnosis.'),('glucose meter continuous monitor record readings', 'A glucose meter measures glucose at a point in time; a continuous glucose monitor estimates glucose over time. Monitoring frequency and targets should be individualized by a care team.')],
'medications': [('insulin metformin medication treatment', 'Diabetes treatments may include insulin or other prescribed medicines, depending on the type of diabetes and personal circumstances. Never start, stop or adjust medicine without your prescribing clinician.'),('missed dose medication insulin', 'For a missed diabetes medicine dose, check the instructions provided with your prescription and contact a pharmacist or prescribing clinician. Do not double a dose unless specifically directed.')]
}
for k,v in KB.items(): print(k,':',len(v),'entries')
def retrieve(query,intent):
    entries = KB[intent]
    mat = vectorizer.transform([clean(t) for t,_ in entries])
    sim = cosine_similarity(vectorizer.transform([clean(query)]),mat)[0]
    idx = int(np.argmax(sim))
    return entries[idx][1],float(sim[idx])

In [ ]:
# Requirements 8–9: integrated reasoning, classification and retrieval
# Rules: emergency, out-of-domain, low confidence, medication safety,
# symptoms caution, general educational disclaimer, and empty input.
EMERGENCY = re.compile(r'\b(unconscious|unresponsive|seizure|seizures|cannot wake|can.t wake|difficulty breathing|trouble breathing|fainting|collapsed|confusion and vomiting|vomiting and confusion|diabetic ketoacidosis|\bdka\b)\b',re.I)
DOMAIN = re.compile(r'\b(diabet\w*|glucose|blood sugar|insulin|a1c|hba1c|metformin|prediabet\w*|glycemi\w*|carbohydrat\w*|sugar level|pancreas|gestational)\b',re.I)
MED_CHANGE = re.compile(r'\b(stop|skip|double|increase|decrease|change|missed|adjust|dose)\b',re.I)
DISCLAIMER = 'This is educational information, not a diagnosis or personalized medical advice.'
def predict_intent(query):
    probabilities = clf.predict_proba(vectorizer.transform([clean(query)]))[0]
    ix = int(np.argmax(probabilities))
    return clf.classes_[ix],float(probabilities[ix])
def assistant(query):
    query = str(query).strip()
    result = {'query':query,'intent':'','confidence':0.,'rule':'','response':''}
    if not query:
        result.update(intent='empty',rule='R1 empty input',response='Please type a diabetes-care question.'); return result
    if EMERGENCY.search(query):
        result.update(intent='emergency',rule='R2 emergency escalation',response='This may be a medical emergency. Call local emergency services now (112 in India), and seek immediate medical attention.'); return result
    intent,conf = predict_intent(query)
    result.update(intent=intent,confidence=round(conf,3))
    if not DOMAIN.search(query):
        result.update(intent='unsupported',rule='R3 unsupported query',response='I cover diabetes care only. Please ask about diabetes symptoms, risks, prevention, nutrition, testing or medicines.'); return result
    if conf < 0.34 or len(preprocess(query)) < 2:
        result.update(intent='uncertain',rule='R4 low confidence',response='I am not confident I understood. Could you rephrase your diabetes-related question with more detail?'); return result
    response,sim = retrieve(query,intent)
    rules = ['R5 knowledge retrieval']
    if intent == 'medications' and MED_CHANGE.search(query):
        response += ' For medication changes or missed doses, check with your clinician or pharmacist.'
        rules.append('R6 medication safety')
    if intent == 'symptoms':
        response += ' If symptoms are severe or rapidly worsening, seek urgent medical care.'
        rules.append('R7 symptom caution')
    response += ' '+DISCLAIMER
    rules.append('R8 disclaimer')
    result.update(rule=' + '.join(rules),response=response,retrieval_similarity=round(sim,3))
    return result
print(assistant('What is an A1C diabetes test?'))

In [ ]:
# Requirement 10: Google Colab interactive interface (supports multiple questions)
import ipywidgets as widgets
from IPython.display import display, HTML
box = widgets.Text(placeholder='Ask a diabetes-care question...',layout=widgets.Layout(width='75%'))
button = widgets.Button(description='Ask',button_style='info')
output = widgets.Output()
def show_answer(question):
    result = assistant(question)
    with output:
        display(HTML('<div style="padding:12px;margin:8px 0;border:1px solid #ddd;border-radius:8px">'
                     '<b>You:</b> '+html.escape(question)+'<br><b>Assistant:</b> '+html.escape(result['response'])+
                     '<br><small>Intent: '+html.escape(result['intent'])+' | Confidence: '+str(result['confidence'])+
                     ' | Rule: '+html.escape(result['rule'])+'</small></div>'))
def on_click(_):
    q=box.value.strip()
    if q: show_answer(q); box.value=''
button.on_click(on_click)
display(HTML('<h3>Diabetes Care Information Assistant</h3>'),widgets.HBox([box,button]),output)
print('If widgets are unavailable, run chat_loop() in the next cell.')

In [ ]:
# Optional text interface fallback
def chat_loop():
    print('Diabetes Assistant (type quit to exit)')
    while True:
        q = input('You: ').strip()
        if q.lower() in ('quit','exit'): break
        print('Assistant:',assistant(q)['response'])
# Uncomment to use: chat_loop()

In [ ]:
# Required test cases: all six intents, paraphrase, unsupported, low-confidence, emergency
TESTS = [
 ('symptoms','Can diabetes make you thirsty?'),
 ('risk_factors','Does diabetes run in families?'),
 ('prevention','How can I prevent type 2 diabetes with exercise?'),
 ('nutrition','What carbohydrates can I eat with diabetes?'),
 ('monitoring','What does an A1C diabetes test show?'),
 ('medications','What is metformin for in diabetes?'),
 ('paraphrase','How do I keep track of my glucose readings?'),
 ('unsupported','How do I fix my laptop screen?'),
 ('low-confidence','diabetes xylophone nebulous orbit'),
 ('emergency','A person with diabetes is unconscious and unresponsive')]
for label,question in TESTS:
    result=assistant(question)
    print(f"[{label}] {question}\n  -> {result['intent']} (confidence {result['confidence']}) | {result['rule']}\n  {result['response']}\n")
print('Note: A low-confidence example may vary with model training; rule coverage can also be tested with a short vague input.')

## Submission checklist
- 60 labelled queries in 6 intent categories
- 4 taxonomy categories, each with at least 2 subcategories
- 13 concepts and 12 relationships
- Lowercasing, tokenization, stopword removal, lemmatization
- TF-IDF, two classifiers, held-out evaluation and confusion matrix
- Knowledge base, 8 reasoning/safety rules, integrated assistant
- Interactive Colab widget, fallback chat, required tests

**Important:** This is a classroom demonstration. The small synthetic training corpus does not establish clinical accuracy. Emergency keyword rules are not exhaustive.